<a href="https://colab.research.google.com/github/VladShajdulin/OTUS/blob/main/home_work_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
! pip install evaluate
! git clone https://github.com/RussianNLP/RuCoLA

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00
Cloning into 'RuCoLA'...
remote: Enumerating objects: 76, done.
remote: Counting objects: 100% (76/76), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 76 (delta 31), reused 52 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (76/76), 948.93 KiB | 3.75 MiB/s, done.
Resolving deltas: 100% (31/31), done.


In [2]:
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import Dataset
from sklearn.model_selection import train_test_split
import evaluate

In [3]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu'
print('Device', device)

Device cuda


In [30]:
train_df = pd.read_csv('/content/RuCoLA/data/in_domain_train.csv')[['sentence', 'acceptable']]
test_df = pd.read_csv('/content/RuCoLA/data/in_domain_dev.csv')[['sentence', 'acceptable']]
train, val = train_test_split(
    train_df,
    test_size=0.2,
    random_state=345,
    shuffle=True,
    stratify=train_df['acceptable']
)
train['acceptable'].mean()

np.float64(0.7451945988880063)

# 1. Обучение BERT

In [31]:
name_bert = 'ai-forever/ruBert-base'
tokenizer = AutoTokenizer.from_pretrained(name_bert)
model = AutoModelForSequenceClassification.from_pretrained(name_bert, num_labels=2).to(device)

train, val = Dataset.from_pandas(train), Dataset.from_pandas(val)
train = train.map(lambda x: tokenizer(x['sentence']), batched=True).rename_column('acceptable', 'label')
val = val.map(lambda x: tokenizer(x['sentence']), batched=True).rename_column('acceptable', 'label')

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors='pt'
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ai-forever/ruBert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/6295 [00:00<?, ? examples/s]

Map:   0%|          | 0/1574 [00:00<?, ? examples/s]

In [33]:
clf_metrics = evaluate.combine(['f1', 'accuracy', 'precision', 'recall'])
def compute_metrics(eval_pred):
  preds, labels = eval_pred

  return clf_metrics.compute(predictions=preds.argmax(axis=1), references=labels)

args = TrainingArguments(
    output_dir='bert_model',
    save_strategy='no',
    eval_strategy='steps',
    eval_steps=100,
    logging_strategy='steps',
    logging_steps=100,

    num_train_epochs=3,
    max_steps=800,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    warmup_ratio=0.1,
    weight_decay=0.01,
    optim='adamw_torch',
    lr_scheduler_type='linear'
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train,
    eval_dataset=val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [34]:
trainer.train()

Step,Training Loss,Validation Loss,F1,Accuracy,Precision,Recall
100,0.579400,0.567722,0.854023,0.745235,0.745235,1.000000
200,0.531600,0.537253,0.858627,0.757942,0.760184,0.986360
300,0.511000,0.534843,0.861607,0.763659,0.764356,0.987212
400,0.494000,0.495178,0.872699,0.789072,0.793031,0.970162
500,0.357500,0.512800,0.856798,0.773825,0.811120,0.907928
600,0.346400,0.555352,0.874519,0.792884,0.797193,0.968457
700,0.317800,0.532182,0.873284,0.794790,0.808866,0.948849
800,0.329700,0.521720,0.871937,0.794155,0.812822,0.940324


TrainOutput(global_step=800, training_loss=0.43344929695129397, metrics={'train_runtime': 147.1935, 'train_samples_per_second': 86.96, 'train_steps_per_second': 5.435, 'total_flos': 191257070585280.0, 'train_loss': 0.43344929695129397, 'epoch': 2.030456852791878})

In [42]:
test_true = torch.from_numpy(test_df['acceptable'].values).to(device)
with torch.no_grad():
  tokens = tokenizer(
      test_df['sentence'].to_list(),
      return_tensors='pt',
      padding=True
  ).to(device)
  preds = model(**tokens).logits.argmax(axis=1)

clf_metrics.compute(predictions=preds, references=test_true)

{'f1': 0.8668769716088328,
 'accuracy': 0.785350966429298,
 'precision': 0.8063380281690141,
 'recall': 0.937244201909959}

In [46]:
text0 = test_df[test_df['acceptable'] == 0].iloc[0, 0]
text0

'У многих туристов, кто посещают Кемер весной, есть шанс застать снег на вершине горы Тахталы и даже сочетать пляжный отдых с горнолыжным.'

In [48]:
with torch.no_grad():
  tokens = tokenizer(
      text0,
      return_tensors='pt',
      padding=True
  ).to(device)
  label = model(**tokens).logits.argmax(axis=1)
print(label.item())

0


F1: 0.867\
Accuracy: 0.785 (не многим лучше, чем предсказание самого частотного класса - 0.745)\
Precision: 0.806\
Recall: 0.937

# 2. GPT